In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# import os, shutil
# from pathlib import Path

# Sorgente dei pesi finali
# SRC_WEIGHTS = '/content/drive/MyDrive/2026_MLinf_gr41/Waste-Project/regnety16gf_acq_mild.pth'

# WORK = Path('/content/submission_test')
# (WORK / 'eval').mkdir(parents=True, exist_ok=True)
# shutil.copy(SRC_WEIGHTS, WORK / 'regnety16gf_acq_mild.pth')

# os.chdir(WORK)
# print('CWD:', os.getcwd())
# print('Contenuto:', os.listdir('.'))

Mounted at /content/drive
CWD: /content/submission_test
Contenuto: ['regnety16gf_acq_mild.pth', 'eval']


In [ ]:
# import random, shutil
# from pathlib import Path
# from PIL import Image
# import numpy as np

# DATASET = Path('/content/drive/MyDrive/2026_MLinf_gr41/Waste-Project/dataset')
# EVAL = Path('/content/submission_test/eval')

# n = 0
# for d in sorted([p for p in DATASET.iterdir() if p.is_dir()]):
    #imgs = [p for p in d.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    #if imgs:
        #shutil.copy(random.choice(imgs), EVAL / f'real_{d.name}.jpg')
        #n += 1
# print(f'Copiate {n} immagini reali')

# Image.fromarray(np.random.randint(0, 255, (180, 240), dtype=np.uint8), mode='L').save(EVAL / 'edge_grayscale.png')

# Image.fromarray(np.random.randint(0, 255, (200, 200, 4), dtype=np.uint8), mode='RGBA').save(EVAL / 'edge_rgba.png')

# Image.fromarray(np.random.randint(0, 255, (37, 300, 3), dtype=np.uint8), mode='RGB').save(EVAL / 'edge_oddsize.jpg')

# print('eval/ contiene:', sorted(os.listdir(EVAL)))

Copiate 8 immagini reali
eval/ contiene: ['edge_grayscale.png', 'edge_oddsize.jpg', 'edge_rgba.png', 'real_battery.jpg', 'real_clothing.jpg', 'real_glass.jpg', 'real_metal.jpg', 'real_organic.jpg', 'real_papery.jpg', 'real_plastic.jpg', 'real_undifferentiated.jpg']


/tmp/ipykernel_1973/623799152.py:17: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(np.random.randint(0, 255, (180, 240), dtype=np.uint8), mode='L').save(EVAL / 'edge_grayscale.png')
/tmp/ipykernel_1973/623799152.py:19: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(np.random.randint(0, 255, (200, 200, 4), dtype=np.uint8), mode='RGBA').save(EVAL / 'edge_rgba.png')
/tmp/ipykernel_1973/623799152.py:21: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(np.random.randint(0, 255, (37, 300, 3), dtype=np.uint8), mode='RGB').save(EVAL / 'edge_oddsize.jpg')


#Before to start
Put this notebook file in the submitted Google Drive directory.

Edit **Section 1**, i.e., the first part of this notebook file implementing:
1. **Section 1.1**: the loading of your best model(s):
  * a. Create the network(s)
  * b. Load the best saved model(s) using a relative path (e.g., `"./model.pth"` or `"./model/best.pth"`)
2. **Section 1.2**: the predict function of your model(s):
  * a. Pre-process the input batch of data
  * b. Perform the forward using the network(s)
  * c. Post-process the output of the network(s)


DO NOT EDIT **Section 2** since it replicates the exact evaluation code we run on the private test set.

To test your model you have to:
* create a directory named `"eval"` containing the test images
* run all cells

#Section 1: YOUR CODE

##Section 1.1: IMPLEMENT HERE THE FUNCTION TO LOAD YOUR MODEL
For example, here we use a simple network.

**IMPORTANT: load the trained weights of your model here!**

In [ ]:
# Sezione 1.1: caricamento del modello

import torch
import torch.nn as nn
from torchvision import models

NUM_CLASSES = 8

# Percorso relativo ai pesi
# Modello finale scelto: RegNetY-1.6GF allenato con la ricetta 'acq_mild'
WEIGHTS_PATH = "./regnety16gf_acq_mild.pth"

def load_model():
    # Stessa architettura usata in training: RegNetY-1.6GF con testa lineare a 8 classi
    # weights=None: non scarichiamo i pesi ImageNet, carichiamo i nostri
    model = models.regnet_y_1_6gf(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

    # Abbiamo salvato lo state_dict in training, quindi lo ricarichiamo così
    state = torch.load(WEIGHTS_PATH, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    model.load_state_dict(state)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    return model

##Section 1.2: IMPLEMENT HERE YOUR PREDICT FUNCTION

Consider that the input is a batch of data with:
```
shape = (batch_size, rows, cols, channels=3)
dtype = uint8
```
For example, here we implement a simple pre and post processing and we call the model forward.

**IMPORTANT: implement your own pre- and post-processing pipeline here!**

Note that you can use torchvision transformations on a single image of the batch. [Read the documentation](https://docs.pytorch.org/vision/main/transforms.html) for more details.

In [ ]:
# Sezione 1.2: funzione predict (con TTA flip orizzontale)

import numpy as np
import torch
import torch.nn.functional as F
from torchvision import transforms

IMG_SIZE = 224
USE_TTA = True                                                                  # Usare 'False' per tornare alla predict a vista singola (senza TTA)

# Preprocessing identico a training/validation
_preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def _to_hwc_rgb(img):
    # Rende ogni immagine (H, W, 3) uint8
    img = np.asarray(img)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    elif img.ndim == 3:
        c = img.shape[-1]
        if c == 1:
            img = np.repeat(img, 3, axis=-1)
        elif c == 4:
            img = img[..., :3]
        elif c != 3:
            img = img[..., :3]
    return img.astype(np.uint8)

def predict(model, X):
    '''
    X: numpy uint8 (batch_size, rows, cols, 3). Ritorna numpy uint8 (batch_size, 1), label 0..7.
    USE_TTA=True: media delle probabilità su immagine originale + flip orizzontale.
    '''
    model.eval()
    device = next(model.parameters()).device

    X = np.asarray(X)
    if X.ndim == 2:
        X = X[None, ..., None]
    elif X.ndim == 3:
        if X.shape[-1] in (1, 3, 4):
            X = X[None, ...]
        else:
            X = X[..., None]

    tensors = [_preprocess(_to_hwc_rgb(X[i])) for i in range(X.shape[0])]
    batch = torch.stack(tensors).to(device)

    with torch.no_grad():
        if USE_TTA:
            flipped = torch.flip(batch, dims=[3])                               # flip orizzontale
            both = torch.cat([batch, flipped], dim=0)
            probs = F.softmax(model(both), dim=1)
            B = batch.shape[0]
            probs = 0.5 * (probs[:B] + probs[B:])                               # media originale + flip
        else:
            probs = F.softmax(model(batch), dim=1)
        preds = probs.argmax(dim=1)

    return preds.detach().cpu().numpy().astype(np.uint8).reshape(-1, 1)

Run this cell to verify that the produced output is well formatted

In [ ]:
import numpy as np
model = load_model()
X = np.random.randint(0, 255, size=(2,256,256,3), dtype=np.uint8) # this batch size is used only as an example
y = predict(model,X)
assert (y.shape == (X.shape[0],1) and y.dtype == np.uint8 \
      and (y >= 0).all() and (y < NUM_CLASSES).all()), "Verify your model loading or predict function."

#Before to run the code
1. eventually change the current working directory using the `os.chdir` function (DO IT IN THE EMPTY CELL BELOW THIS ONE)
2. create a directory named `"./eval"` in the current working directory
3. verify that the `"./eval"` directory contains the image files for the test

Then, run all the cells.

#Section 2: Test code (DO NOT MODIFY THE CODE BELOW!)
This is exactly the code we run for the final test.

Just run the code to verify that it works. This is the exact code we run for the final evaluation on the private test set.

In [ ]:
#from google.colab import drive;drive.mount('/content/drive'); import os; os.chdir("/content/drive/My Drive/Didattica/ML/exam_2025_2026/project_work_Ing_Inf/evaluation")

In [ ]:
import os
from glob import glob

test_dir = "./eval/" # Do not modify this path, instead create a directory with this name in the same folder of this test.ipynb file
assert os.path.isdir(test_dir), "The evaluation directory does not exist. Create it, put some images in it and run again this cell."

samples = [sample for sample in glob(test_dir + '/**', recursive=True) if os.path.isfile(sample)]
assert len(samples) > 0, "The evaluation directory is empty. Put some images in it and run again this cell."
print(f"Found {len(samples)} samples in {test_dir}")

Found 11 samples in ./eval/


In [ ]:
import os
from PIL import Image
import numpy as np
from tqdm import tqdm

# Run YOUR LOAD_MODEL FUNCTION
model = load_model()

# Main loop
verbose = True

PREDICTIONs = np.zeros((len(samples), )) * np.nan
for i, img_path in tqdm(enumerate(samples), desc="Processing samples"):
  try:  # ATTENTION: any error occurring in this try-catch means that the corresponding PREDICTION is evaluated as an ERROR of the neural network
    # Open images
    rgb_image = Image.open(img_path)
    rgb_array = np.asarray(rgb_image)[None, ...]
    if verbose:
      print(f"\n  Loaded {img_path}. The input batch has shape {rgb_array.shape} and dtype {rgb_array.dtype}")

    # Run YOUR PREDICT FUNCTION
    predicted_labels_array = predict(model, rgb_array).squeeze()
    if verbose:
      print(f"  Predicted label {predicted_labels_array}")

    PREDICTIONs[i] = predicted_labels_array

  except FileNotFoundError:
    print(f"  Error: Could not find image file {img_path}")
  except Exception as e:
    print(f"  Error processing image {img_path}: {e}")

PREDICTIONs = PREDICTIONs.astype(np.uint8)
print(f"Predictions: {PREDICTIONs}")
np.save("predictions.npy", PREDICTIONs)

Processing samples: 2it [00:00, 18.01it/s]


  Loaded ./eval/edge_grayscale.png. The input batch has shape (1, 180, 240) and dtype uint8
  Predicted label 5

  Loaded ./eval/real_organic.jpg. The input batch has shape (1, 194, 259, 3) and dtype uint8
  Predicted label 4

  Loaded ./eval/edge_rgba.png. The input batch has shape (1, 200, 200, 4) and dtype uint8
  Predicted label 2

  Loaded ./eval/real_metal.jpg. The input batch has shape (1, 384, 512, 3) and dtype uint8
  Predicted label 3

  Loaded ./eval/real_papery.jpg. The input batch has shape (1, 177, 284, 3) and dtype uint8
  Predicted label 5

  Loaded ./eval/real_plastic.jpg. The input batch has shape (1, 225, 225, 3) and dtype uint8


Processing samples: 11it [00:00, 29.96it/s]

  Predicted label 6

  Loaded ./eval/real_battery.jpg. The input batch has shape (1, 194, 259, 3) and dtype uint8
  Predicted label 0

  Loaded ./eval/real_clothing.jpg. The input batch has shape (1, 400, 533, 3) and dtype uint8
  Predicted label 1

  Loaded ./eval/real_undifferentiated.jpg. The input batch has shape (1, 183, 276, 3) and dtype uint8
  Predicted label 7

  Loaded ./eval/real_glass.jpg. The input batch has shape (1, 200, 200, 3) and dtype uint8
  Predicted label 2

  Loaded ./eval/edge_oddsize.jpg. The input batch has shape (1, 37, 300, 3) and dtype uint8
  Predicted label 5
Predictions: [5 4 2 3 5 6 0 1 7 2 5]
